# 📊 Notebook 03: Análise Exploratória, Inferência Estatística e Modelagem Preditiva
### Projeto: Análise e Diagnóstico de Acidentes de Trânsito em Bauru/SP (2015–2026)

Neste notebook desenvolvemos as análises centrais de segurança viária para Bauru:
1. Evolução histórica de sinistros e vítimas fatais (2015–2026);
2. Distribuição temporal (dias da semana e horários de pico);
3. Teste de Hipótese Estatística (*Student's t-test*) entre dias úteis e finais de semana;
4. Letalidade comparativa por modal e tipo de choque/colisão;
5. Matriz de calor (*Heatmap*) dia vs hora;
6. Modelo Preditivo Supervisionado (**Random Forest**) com *class weights* e avaliação de *Feature Importance* (Gini).

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

df = pd.read_csv('../data/bauru_limpo.csv', sep=';', encoding='utf-8')
print(f"Dataset carregado: {len(df)} ocorrências.")

## 1. Evolução Histórica Anual (2015–2026)

In [ ]:
resumo_anual = df.groupby('ano').agg(
    total_sinistros=('id_sinistro', 'count'),
    obitos=('obitos', 'sum'),
    feridos_graves=('feridos_graves', 'sum'),
    feridos_leves=('feridos_leves', 'sum')
).reset_index()

display(resumo_anual)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(resumo_anual['ano'], resumo_anual['total_sinistros'], color='#2b5c8f', alpha=0.85, width=0.6, label='Total Sinistros')
ax1.set_xlabel('Ano')
ax1.set_ylabel('Total de Sinistros', color='#2b5c8f')
ax1.set_xticks(resumo_anual['ano'])

ax2 = ax1.twinx()
ax2.plot(resumo_anual['ano'], resumo_anual['obitos'], color='#d9383a', linewidth=3, marker='o', label='Óbitos')
ax2.set_ylabel('Óbitos Confirmados', color='#d9383a')
ax2.grid(False)

plt.title('Evolução Anual de Sinistros e Óbitos em Bauru/SP (2015–2026)')
plt.tight_layout()
plt.show()

## 2. Teste de Hipótese Estatística: Dias Úteis vs Fins de Semana
Avaliando se a média diária de acidentes difere estatisticamente entre os dias de semana (segunda a sexta) e fins de semana (sábado e domingo).

In [ ]:
df_completo = df[(df['ano'] >= 2019) & (df['ano'] <= 2025)].copy()
df_completo['data_dt'] = pd.to_datetime(df_completo['data_sinistro_dt'])
acidentes_dia = df_completo.groupby('data_dt').agg(
    total=('id_sinistro', 'count'),
    fim_de_semana=('is_fim_de_semana', 'first')
).reset_index()

uteis = acidentes_dia[acidentes_dia['fim_de_semana'] == 0]['total']
finais = acidentes_dia[acidentes_dia['fim_de_semana'] == 1]['total']

t_stat, p_val = stats.ttest_ind(uteis, finais, equal_var=False)
print(f"Média em Dias Úteis: {uteis.mean():.2f} acidentes/dia (std: {uteis.std():.2f})")
print(f"Média em Fins de Semana: {finais.mean():.2f} acidentes/dia (std: {finais.std():.2f})")
print(f"Estatística t = {t_stat:.3f}, p-valor = {p_val:.4f}")
if p_val > 0.05:
    print("Conclusão: Não há diferença estatisticamente significativa na média diária (p > 0.05).")
else:
    print("Conclusão: Diferença estatisticamente significativa detectada.")

## 3. Letalidade por Tipo de Sinistro e Modal Envolvido

In [ ]:
tabela_tipo = df.groupby('tipo_sinistro_simplificado').agg(
    total=('id_sinistro', 'count'),
    obitos=('obitos', 'sum'),
    feridos_graves=('feridos_graves', 'sum')
).reset_index()
tabela_tipo['letalidade_pct'] = (tabela_tipo['obitos'] / tabela_tipo['total']) * 100
tabela_tipo = tabela_tipo.sort_values(by='letalidade_pct', ascending=False)
display(tabela_tipo)

## 4. Matriz de Calor: Horário do Dia vs Dia da Semana

In [ ]:
ordem_dias = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira', 'Sexta-feira', 'Sábado', 'Domingo']
heatmap_data = df[df['hora_num'].notnull()].pivot_table(
    index='dia_da_semana_limpo',
    columns='hora_num',
    values='id_sinistro',
    aggfunc='count',
    fill_value=0
).reindex(ordem_dias)

plt.figure(figsize=(14, 5))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=True, fmt='d', linewidths=0.5)
plt.title('Matriz de Calor: Sinistros por Dia da Semana e Hora do Dia (Bauru/SP)')
plt.xlabel('Hora do Dia (0h - 23h)')
plt.ylabel('Dia da Semana')
plt.tight_layout()
plt.show()

## 5. Modelagem Supervisionada de Gravidade (Random Forest)

In [ ]:
df_ml = df[df['hora_num'].notnull()].copy()
features_cols = ['envolve_moto', 'envolve_auto', 'envolve_pedestre', 'envolve_bicicleta', 'envolve_pesado', 'is_fim_de_semana']

dummies_via = pd.get_dummies(df_ml['tipo_via_simplificado'], prefix='via', drop_first=True)
dummies_tipo = pd.get_dummies(df_ml['tipo_sinistro_simplificado'], prefix='tipo', drop_first=True)
dummies_turno = pd.get_dummies(df_ml['turno_ajustado'], prefix='turno', drop_first=True)

X = pd.concat([df_ml[features_cols], df_ml[['hora_num']], dummies_via, dummies_tipo, dummies_turno], axis=1)
y = df_ml['severidade_grave_ou_fatal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Acurácia:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred))

importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(9, 5))
sns.barplot(x=importancias.head(10).values, y=importancias.head(10).index, hue=importancias.head(10).index, palette='mako', legend=False)
plt.title('Top 10 Variáveis Mais Determinantes para a Gravidade do Sinistro (Bauru)')
plt.xlabel('Importância Relativa (Gini)')
plt.tight_layout()
plt.show()